In [71]:
import numpy as np
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
from sklearn.datasets import make_blobs, load_iris # เรียกใช้ข้อมูลที่มากับ SciKit Learn
import pandas as pd

In [72]:
restaurant = pd.read_csv("Restaurant.csv")
restaurant

,Order ID,Date,Product,Price,Quantity,Purchase Type,Payment Method,Manager,City
0,10452,07-11-2022,Fries,3.49,573.07,Online,Gift Card,Tom Jackson,London
1,10453,07-11-2022,Beverages,2.95,745.76,Online,Gift Card,Pablo Perez,Madrid
2,10454,07-11-2022,Sides & Other,4.99,200.40,In-store,Gift Card,Joao Silva,Lisbon
3,10455,08-11-2022,Burgers,12.99,569.67,In-store,Credit Card,Walter Muller,Berlin
4,10456,08-11-2022,Chicken Sandwiches,9.95,201.01,In-store,Credit Card,Walter Muller,Berlin
...,...,...,...,...,...,...,...,...,...
249,10709,28-12-2022,Sides & Other,4.99,200.40,Drive-thru,Gift Card,Walter Muller,Berlin
250,10710,29-12-2022,Burgers,12.99,754.43,Drive-thru,Gift Card,Walter Muller,Berlin
251,10711,29-12-2022,Chicken Sandwiches,9.95,281.41,Drive-thru,Gift Card,Walter Muller,Berlin
252,10712,29-12-2022,Fries,3.49,630.37,Drive-thru,Gift Card,Walter Muller,Berlin


In [73]:
#ข้อมูลใน csv
restaurant.info()                                      # to show some basic informations about the dataset

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 254 entries, 0 to 253
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Order ID        254 non-null    int64  
 1   Date            254 non-null    object 
 2   Product         254 non-null    object 
 3   Price           254 non-null    float64
 4   Quantity        254 non-null    float64
 5   Purchase Type   254 non-null    object 
 6   Payment Method  254 non-null    object 
 7   Manager         254 non-null    object 
 8   City            254 non-null    object 
dtypes: float64(2), int64(1), object(6)
memory usage: 18.0+ KB


In [74]:
#ตรวจสอบ missing values,หาค่าสถิติพื้นฐาน
restaurant[restaurant.isnull().any(axis=1)].head()
#restaurant.describe()

,Order ID,Date,Product,Price,Quantity,Purchase Type,Payment Method,Manager,City


In [75]:
#ตรวจหาข้อมูลซ้ำ
restaurant.duplicated().sum()

np.int64(0)

In [76]:
#check ชนิดข้อมูล
restaurant.dtypes

Order ID            int64
Date               object
Product            object
Price             float64
Quantity          float64
Purchase Type      object
Payment Method     object
Manager            object
City               object
dtype: object

In [77]:
#เช็คว่าควรเป็นตัวอักษร หรือตัวเลข
#restaurant["Product"] = pd.to_numeric(restaurant["Product"], errors="coerce")
#restaurant["Price"] = pd.to_numeric(restaurant["Price"], errors="coerce")
#restaurant["Quantity"] = pd.to_numeric(restaurant["Quantity"], errors="coerce")
#restaurant["Purchase Type"] = pd.to_numeric(restaurant["Purchase Type"], errors="coerce")
#restaurant["Payment Method"] = pd.to_numeric(restaurant["Payment Method"], errors="coerce")
#restaurant["Manager"] = pd.to_numeric(restaurant["Manager"], errors="coerce")
#restaurant["City"] = pd.to_numeric(restaurant["City"], errors="coerce")

In [78]:
Q1 = restaurant["Price"].quantile(0.25)
Q3 = restaurant["Price"].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = restaurant[
    (restaurant["Price"] < lower) |
    (restaurant["Price"] > upper)
]

In [79]:
outliers

,Order ID,Date,Product,Price,Quantity,Purchase Type,Payment Method,Manager,City
28,10482,13-11-2022,Fries,25.50,630.37,In-store,Credit Card,Joao Silva,Lisbon
29,10486,14-11-2022,Chicken Sandwiches,29.05,201.01,In-store,Credit Card,Joao Silva,Lisbon


In [80]:
#ปรับค่าให้เขียนตรงกันเช็คค่าที่ไม่ซ้ำกัน
print(restaurant["Product"].unique())
print(restaurant["Price"].unique())
print(restaurant["Quantity"].unique())
print(restaurant["Purchase Type"].unique())
print(restaurant["Payment Method"].unique())
print(restaurant["Manager"].unique())
print(restaurant["City"].unique())

['Fries' 'Beverages' 'Sides & Other' 'Burgers' 'Chicken Sandwiches']
[ 3.49  2.95  4.99 12.99  9.95 25.5  29.05]
[573.07 745.76 200.4  569.67 201.01 554.27 677.97 630.37 523.48 508.08
 538.88 687.68 477.29 492.69 461.89 446.5  585.07 221.11 600.46 631.25
 646.65 677.44 241.21 261.31 692.84 281.41 723.63 301.51 754.43]
['Online ' 'In-store ' 'Drive-thru ']
[' Gift Card' ' Credit Card' ' Cash']
['Tom      Jackson' '       Pablo Perez' 'Joao    Silva' 'Walter Muller'
 'Remy    Monet' 'Remy Monet' '       Remy Monet' 'Remy     Monet'
 'Pablo Perez' 'Pablo   Perez' 'Pablo  Perez' 'Pablo    Perez'
 'Joao Silva' 'Tom Jackson']
['London' 'Madrid' 'Lisbon' 'Berlin' 'Paris']


In [81]:
#ลบช่องว่างส่วนเกิน

restaurant["Product"] = restaurant["Product"].str.strip()
restaurant["Purchase Type"] = restaurant["Purchase Type"].str.strip()
restaurant["Payment Method"] = restaurant["Payment Method"].str.strip()
restaurant["Manager"] = restaurant["Manager"].str.strip()
restaurant["City"] = restaurant["City"].str.strip()

In [82]:
print(restaurant["Product"].value_counts())
print(restaurant["Price"].value_counts())
print(restaurant["Quantity"].value_counts())
print(restaurant["Purchase Type"].value_counts())
print(restaurant["Payment Method"].value_counts())
print(restaurant["Manager"].value_counts())
print(restaurant["City"].value_counts())

Product
Burgers               52
Chicken Sandwiches    52
Fries                 51
Beverages             50
Sides & Other         49
Name: count, dtype: int64
Price
12.99    52
9.95     51
3.49     50
2.95     50
4.99     49
25.50     1
29.05     1
Name: count, dtype: int64
Quantity
200.40    49
201.01    36
630.37    35
677.97    34
745.76    16
573.07     9
221.11     8
687.68     7
523.48     6
569.67     6
538.88     5
554.27     5
477.29     5
508.08     4
677.44     4
281.41     3
241.21     3
461.89     3
492.69     3
646.65     2
585.07     2
692.84     2
600.46     1
631.25     1
446.50     1
261.31     1
723.63     1
301.51     1
754.43     1
Name: count, dtype: int64
Purchase Type
Online        107
In-store       86
Drive-thru     61
Name: count, dtype: int64
Payment Method
Credit Card    120
Cash            76
Gift Card       58
Name: count, dtype: int64
Manager
Tom Jackson         74
Joao Silva          70
Pablo Perez         43
Walter Muller       30
Remy Monet          2

In [83]:
#พิมพ์ตัวเล็ก-ใหญ่ให้เหมือนกัน
restaurant["Column"] = restaurant["Column"].str.lower()


KeyError: 'Column'

In [ ]:
# Convert Date column to datetime
restaurant['Date'] = pd.to_datetime(restaurant['Date'], format='%d-%m-%Y')

# Create additional date features
restaurant['Year'] = restaurant['Date'].dt.year
restaurant['Month'] = restaurant['Date'].dt.month
restaurant['Day'] = restaurant['Date'].dt.day
restaurant['DayOfWeek'] = restaurant['Date'].dt.dayofweek  # 0=Monday, 6=Sunday
restaurant['WeekOfYear'] = restaurant['Date'].dt.isocalendar().week

# Calculate total revenue per order
restaurant['Total_Revenue'] = restaurant['Price'] * restaurant['Quantity']

print("=== DATE RANGE ===")
print(f"From: {restaurant['Date'].min()} to {restaurant['Date'].max()}")
print(f"Total days: {(restaurant['Date'].max() - restaurant['Date'].min()).days + 1}")